In [1]:
import pandas as pd
import numpy as np

from astropathdb import AstroDB

c:\Users\Michael\miniconda3\envs\astroHome\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(


# Functions

In [ ]:
def format_axis(db, table, phenos, region, size):

    data = db.query(f"SELECT * from {table}").sort_values(["sampleid", "Total_Count"], ascending=[True, False]).reset_index(drop=True)

    data["config_id"] = data[phenos].astype(str).agg("_".join, axis=1)

    data = data[data["config_id"] != "0_0_0_0_0"].reset_index(drop=True)

    data = data.drop(columns=["Other", *phenos])

    data["config"] = data["config_id"]

    data["region"] = region

    data["config_id"] = data["config_id"] + f"_{region}_{size}"

    return data

In [ ]:
def format_den(df, area, clin=None):

    df_formatted = df.copy()

    df_formatted = df_formatted.merge(area, on=["sampleid", "region"], how="left")

    df_formatted = df_formatted.drop(columns=["region"])

    df_formatted["density"] = df_formatted["Total_Count"] / df_formatted["area"]

    df_formatted = df_formatted.replace([np.inf, -np.inf], 0)

    df_formatted = df_formatted.pivot_table(index="config_id", columns="sampleid", values="density").reset_index().fillna(0)

    df_formatted.columns = ["Features", *[f"sampleid_{x}" for x in df_formatted.columns[1:]]]

    if clin is not None:

        df_formatted = pd.concat([clin, df_formatted]).reset_index(drop=True)

    return df_formatted

In [ ]:
def get_area(db, sampleids, nur=8000.0000 / (1.004 * 1.344)):

    area_sql = f"""
        SELECT
            rc.sampleid,
            COUNT(*) / {nur} area
        FROM dbo.randomcell rc
        where (tdist <= 0)
        and rc.sampleid in ({','.join(map(str, sampleids))})
        GROUP BY rc.sampleid
        ORDER BY rc.sampleid
        """
    area_tum = db.query(area_sql)

    area_tum["region"] = "tumor"

    area_sql = f"""
        SELECT
            rc.sampleid,
            COUNT(*) / {nur} area
        FROM dbo.randomcell rc
        where (tdist > 0)
        and rc.sampleid in ({','.join(map(str, sampleids))})
        GROUP BY rc.sampleid
        ORDER BY rc.sampleid
        """
    area_back = db.query(area_sql)

    area_back["region"] = "background"

    area = pd.concat([area_tum, area_back], ignore_index=True)

    area_total = area.groupby("sampleid")["area"].sum().reset_index()

    area_total["region"] = "total"

    area = pd.concat([area, area_total], ignore_index=True)

    return area

In [ ]:
def get_configurations(db, tumour_table, background_table, phenos, size):

    tum = format_axis(db, tumour_table, phenos, "tumor", size)

    back = format_axis(db, background_table, phenos, "background", size)

    total = pd.concat([tum, back], ignore_index=True).groupby(["sampleid", "config"])["Total_Count"].sum().reset_index()

    total["config_id"] = total["config"] + f"_total_{size}"

    total = total.drop(columns=["config"])

    tum = tum.drop(columns=["config"])

    back = back.drop(columns=["config"])

    combined = pd.concat([tum, back, total], ignore_index=True).fillna("total")

    area = get_area(db, combined["sampleid"].unique())

    combined = format_den(combined, area)

    return combined

In [ ]:
def lung_norm(data, lung_data):

    lung = lung_data.copy()

    configs = lung[1:]["Features"]

    data = data[data["Features"].isin(configs)].reset_index(drop=True)

    data = data.merge(lung["Features"][1:], on="Features", how="outer").fillna(0)

    param_dict = {}
    for config in configs:

        mean = lung[lung["Features"] == config].iloc[:, 1:].mean(axis=1).values[0]
        std = lung[lung["Features"] == config].iloc[:, 1:].std(axis=1).values[0]

        params = {config: {"mean": mean, "std": std}}

        param_dict.update(params)
    

    cols = data.columns[1:]

    for config in data["Features"]:

        data.loc[data["Features"] == config, cols] = (data.loc[data["Features"] == config, cols] - param_dict[config]["mean"]) / param_dict[config]["std"]
    
    return data

In [ ]:
def compute_SCI(data, weights):

    data = data.copy()

    data_t = data.T.reset_index()

    data_t.columns = data_t.iloc[0]

    data_t = data_t.drop(index=0)
    data_t = data_t.reset_index(drop=True)

    weights = weights.copy()
    
    weights = weights.set_index("configuration")["weight"]

    data_weighted = data_t[weights.index] * weights.values

    data_t.loc[:, f"OS_composite_score"] = data_weighted.sum(axis=1)

    data_t = data_t[["Features", f"OS_composite_score"]]

    data_t = data_t.rename(columns={"Features": "sampleid"})

    data_t["sampleid"] = [int(x.split("_")[1]) for x in data_t["sampleid"]]

    data_t = data_t.sort_values(f"OS_composite_score", ascending=False).reset_index(drop=True)

    return data_t

# Get Spatial Configuration Densities Per Specimen

## WSI01

In [ ]:
# Get configurations for all distances

db01 = AstroDB(database="wsi01")

# 10 microns
configs_10 = get_configurations(db01, "test.dbo.wsi01_TumorConfigs_10", "test.dbo.wsi01_BackgroundConfigs_10", ["CD8", "FoxP3", "Tumor", "CD68", "CD8FoxP3"], "10")

# 17.5 microns
configs_175 = get_configurations(db01, "test.dbo.wsi01_TumorConfigs_175", "test.dbo.wsi01_BackgroundConfigs_175", ["CD8", "FoxP3", "Tumor", "CD68", "CD8FoxP3"], "175")

# 25 microns
configs_25 = get_configurations(db01, "test.dbo.wsi01_TumorConfigs_25", "test.dbo.wsi01_BackgroundConfigs_25", ["CD8", "FoxP3", "Tumor", "CD68", "CD8FoxP3"], "25")

configs = pd.concat([configs_10, configs_175, configs_25], ignore_index=True)


# Normalize to Lung Cohort
lung_data = pd.read_excel("../data/inputs/pre_lin_filt_OS_results.xlsx", sheet_name="RankedCellData")

configs_norm = lung_norm(configs, lung_data)


# Compute spatial configuration index
weights = pd.read_excel("../data/inputs/pre_lin_filt_OS_results.xlsx", sheet_name="Weights")

indices = compute_SCI(configs_norm, weights)